# Stall Approach & Recovery — Load-Factor `Aircraft`

Self-contained notebook exploring the small-UAS load-factor model at the edge of the lift
envelope, in two scenarios:

1. **Power-off stall** — level flight at idle (power deficit); the aircraft decelerates, angle of
   attack rises until `CL_eff` reaches `CL_max`, the commanded load factor can no longer be held,
   and the aircraft *mushes* (sinks) — then recovers as speed rebuilds.
2. **Power-on stall** — full throttle with a sustained pull-up; the steepening climb bleeds
   airspeed into the same `CL_max` limit, the aircraft departs/sinks, then recovers.

**Model note.** The load-factor `Aircraft` is *CL_max-limited* (the FBW α-limiter / allocator
caps α at the stall peak): it reaches `CL_max`, cannot sustain the load factor, and mushes — but
it does **not** enter the separated post-stall branch. So `is_stalled` / `is_cl_recovering` stay
`False` throughout: this notebook shows *approach-to-CL_max and recovery*, not a flown
aerodynamic departure. A flown departure into the separated regime (which would trigger the CL
rate-limited recovery) is the deferred open question **OQ-AC-1** in `docs/design/aircraft.md`.

In [ ]:
# Imports and path setup
import os, sys, json, math, copy
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

_PYTHON_DIR = Path('..').resolve()          # python/ is the parent of notebooks/
if str(_PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(_PYTHON_DIR))
if sys.platform == 'win32':
    os.add_dll_directory(r'C:\msys64\ucrt64\bin')

import liteaero_sim_py as sim
print('liteaero_sim_py:', getattr(sim, '__file__', _PYTHON_DIR))

In [ ]:
# Configuration and the scenario runner
_REPO_ROOT = _PYTHON_DIR.parent
base_cfg = json.load(open(_REPO_ROOT / 'configs' / 'small_uas_ksba.json'))

S_REF    = base_cfg['aircraft']['S_ref_m2']
CL_ALPHA = base_cfg['lift_curve']['cl_alpha']
CL_MAX   = base_cfg['lift_curve']['cl_max']
MASS_KG  = base_cfg['inertia']['mass_kg']
W_N      = MASS_KG * 9.80665
RHO      = 1.225
V_STALL  = math.sqrt(2 * W_N / (RHO * S_REF * CL_MAX))
print(f'V_stall = {V_STALL:.2f} m/s   CL_max = {CL_MAX:.2f}   W = {W_N:.1f} N')

def run_scenario(throttle_fn, nz_fn, V0_mps, duration_s=16.0, dt=0.01, alt0_m=300.0):
    # Run an airborne (no-terrain) scenario; return per-step arrays.
    # throttle_fn(t), nz_fn(t) are command schedules; the aircraft starts trimmed in
    # level flight at V0_mps. No terrain is set, so this is pure aerodynamics.
    cfg = copy.deepcopy(base_cfg)
    q0 = 0.5 * RHO * V0_mps * V0_mps
    a_trim = (W_N / (q0 * S_REF)) / CL_ALPHA          # level-flight trim alpha
    cfg['initial_state'].update({
        'altitude_m': alt0_m, 'pitch_rad': a_trim,
        'velocity_north_mps': V0_mps, 'velocity_east_mps': 0.0, 'velocity_down_mps': 0.0,
    })
    ac  = sim.Aircraft(json.dumps(cfg), dt_s=dt)       # no set_terrain -> free air
    cmd = sim.AircraftCommand()
    N = int(duration_s / dt)
    keys = ['t', 'V', 'alpha_deg', 'cl_eff', 'nz_cmd', 'sink', 'alt', 'pitch_deg',
            'stalled', 'recovering']
    r = {k: np.zeros(N) for k in keys}
    for i in range(N):
        t = ac.state().time_s
        cmd.throttle_nd = throttle_fn(t)
        cmd.n_z         = nz_fn(t)
        ac.step(cmd, dt_s=dt, rho_kgm3=RHO)
        s = ac.state()
        r['t'][i] = t; r['V'][i] = s.airspeed_m_s; r['alpha_deg'][i] = math.degrees(s.alpha_rad)
        r['cl_eff'][i] = ac.cl_eff; r['nz_cmd'][i] = cmd.n_z; r['sink'][i] = s.velocity_down_mps
        r['alt'][i] = s.altitude_m; r['pitch_deg'][i] = math.degrees(s.pitch_rad)
        r['stalled'][i] = ac.is_stalled; r['recovering'][i] = ac.is_cl_recovering
    return r

def plot_scenario(r, title):
    fig, ax = plt.subplots(5, 1, figsize=(9, 11), sharex=True)
    cap = r['cl_eff'] >= 0.99 * CL_MAX                 # CL_max-limited region
    def shade(a):
        a.fill_between(r['t'], 0, 1, where=cap, transform=a.get_xaxis_transform(),
                       color='orange', alpha=0.12, label='CL_max-limited')
    ax[0].plot(r['t'], r['V'] / V_STALL); ax[0].axhline(1.0, color='r', ls='--', lw=1, label='V_stall')
    ax[0].set_ylabel('V / V_stall'); ax[0].legend(loc='upper right')
    ax[1].plot(r['t'], r['alpha_deg']); ax[1].set_ylabel('alpha (deg)')
    ax[2].plot(r['t'], r['cl_eff']); ax[2].axhline(CL_MAX, color='r', ls='--', lw=1, label='CL_max')
    ax[2].set_ylabel('CL_eff'); ax[2].legend(loc='upper right')
    ax[3].plot(r['t'], r['sink']); ax[3].axhline(0, color='k', lw=0.6)
    ax[3].set_ylabel('sink rate\n(m/s, + down)')
    ax[4].plot(r['t'], r['alt']); ax[4].set_ylabel('altitude (m)'); ax[4].set_xlabel('time (s)')
    for a in ax:
        shade(a); a.grid(alpha=0.3)
    fig.suptitle(title); fig.tight_layout()
    print('is_stalled ever:', bool(r['stalled'].any()),
          '  is_cl_recovering ever:', bool(r['recovering'].any()),
          f'  min V/V_stall: {r["V"].min() / V_STALL:.2f}',
          f'  max alpha: {r["alpha_deg"].max():.1f} deg')
    return fig

## 1. Power-off stall

Idle throttle, command level flight (`n_z = 1`). With no thrust the aircraft decelerates; α rises
to hold lift = weight until `CL_eff` reaches `CL_max` near `V_stall`. Below that it can no longer
hold altitude and mushes (positive sink), then recovers as the nose drops and speed rebuilds.

In [ ]:
po = run_scenario(throttle_fn=lambda t: 0.0,
                  nz_fn=lambda t: 1.0,
                  V0_mps=1.4 * V_STALL,
                  duration_s=16.0)
fig = plot_scenario(po, 'Power-off stall - idle, hold level (n_z = 1)')
plt.show()

## 2. Power-on stall

Full throttle with a sustained pull-up (`n_z = 1.6` after 2 s). The steepening climb bleeds
airspeed into the `CL_max` limit at high pitch; the aircraft departs/sinks, then recovers.

In [ ]:
pn = run_scenario(throttle_fn=lambda t: 1.0,
                  nz_fn=lambda t: 1.6 if t > 2.0 else 1.0,
                  V0_mps=1.5 * V_STALL,
                  duration_s=16.0)
fig = plot_scenario(pn, 'Power-on stall - full throttle, sustained pull-up (n_z = 1.6)')
plt.show()

## Discussion

In both scenarios `CL_eff` rises to `CL_max` (shaded region) and the aircraft can no longer
sustain the commanded load factor — it mushes/sinks — then recovers as airspeed rebuilds and α
falls back below the peak. Throughout, `is_stalled` and `is_cl_recovering` remain `False`: the
load-factor model is **CL_max-limited / envelope-protected** and does not enter the separated
post-stall branch, so the CL rate-limited recovery machinery is not exercised here.

Modeling a *flown departure* into the separated regime (which would trip `is_stalled` and the
`cl_recovering` rate limit) is the deferred open question **OQ-AC-1** in
[`docs/design/aircraft.md`](../../docs/design/aircraft.md). Once resolved, this notebook can be
extended with `alpha_max` raised above the stall peak to show a true departure and the
rate-limited lift recovery.